In [ ]:
%pip install sqlglot

In [ ]:
import csv
import pandas as pd
import sqlglot
from sqlglot import exp

In [ ]:
INPUT_FILE = 'asset/result.csv'
CLEANED_FILE = 'asset/cleaned_result.csv'

In [ ]:
with open(INPUT_FILE, 'r', encoding='utf-8') as f_in, \
     open(CLEANED_FILE, 'w', encoding='utf-8', newline='') as f_out:
    
    # Reader này sẽ tự động xử lý các dòng có nội dung nằm trong ngoặc kép
    reader = csv.reader(f_in, quotechar='"', delimiter=',', 
                        skipinitialspace=True, quoting=csv.QUOTE_MINIMAL)
    writer = csv.writer(f_out)
    
    for row in reader:
        # Loại bỏ các ký tự xuống dòng dư thừa bên trong nội dung SQL để làm sạch
        clean_row = [col.replace('\n', ' ').replace('\r', ' ').strip() if isinstance(col, str) else col for col in row]
        writer.writerow(clean_row)

In [ ]:
input = pd.read_csv(CLEANED_FILE,
    on_bad_lines='warn', 
    lineterminator='\n',
    dtype=str, 
    low_memory=False)

In [ ]:
print(len(input))

In [ ]:
print(input['statement'].unique()[1:10])
unique_statments = list(input['statement'].unique())
print(f"Total unique statements: {len(unique_statments)}")

In [ ]:
# check for syntax errors and delete by sqlglot
def is_valid_sql(statement):
    if not isinstance(statement, str):
        return False
    
    try:
        sqlglot.parse_one(statement, read="tsql")
        return True
    except Exception:
        return False

In [ ]:
for statement in unique_statments:
    if not is_valid_sql(statement):
        unique_statments.remove(statement)

In [ ]:
print(len(unique_statments))

In [ ]:
ast_statements = []

In [ ]:
print(sqlglot.parse_one('SELECT p.ra AS pra, p.dec AS pdec, p.type, p.u, p.g, p.r, p.i, p.z, p.psfMag_u, p.psfMag_g, p.psfMag_r, p.psfMag_i, p.psfMag_z, s.z AS redshift, s.class AS spec_class, s.zWarning FROM PhotoObjAll AS p JOIN SpecObjAll AS s ON p.objID = s.bestObjID WHERE p.ra BETWEEN 186.43365814387732 AND 186.43449147721066 AND p.dec BETWEEN 33.738606352550406 AND 33.73943968588374 AND s.zWarning = 0', read="tsql").dump())

In [ ]:
index = 0
for statement in unique_statments:
    ast = sqlglot.parse_one(statement, read="tsql")

    features = {
        "tables": [t.name.lower() for t in ast.find_all(exp.Table)],
        
        # TRÍCH XUẤT CỘT TRONG JOIN ON
        "join_columns": [
                c.name.lower() 
                for j in ast.find_all(exp.Join) 
                if j.args.get("on") 
                for c in j.args["on"].find_all(exp.Column)
        ],

        "columns_where": [c.name.lower() for c in ast.find(exp.Where).find_all(exp.Column)] if ast.find(exp.Where) else [],
        "columns_group": [c.name.lower() for c in ast.find(exp.Group).find_all(exp.Column)] if ast.find(exp.Group) else [],
        "join_types": [j.args.get("side") for j in ast.find_all(exp.Join) if j.args.get("side")],
        "timestamp": index
   }

    index += 1

    ast_statements.append(features)

In [ ]:
print(ast_statements[:5])

In [ ]:
import json

with open('ast_statements.json', 'w') as f:
    json.dump(ast_statements, f, indent=4)
    